[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bernardorivas/wael/blob/main/notebooks/example3_repressilator.ipynb)

# Example 3. Repressilator

Given a parameter sample in DSGRN node 13, this notebook simulates the repressilator as a Hill system and, after adding noise, infers the parameters back with least squares and a PINN. Recovery is evaluated by DSGRN region membership.

In [ ]:
%pip install -q DSGRN
%pip install -q tqdm git+https://github.com/marciogameiro/DSGRN_utils.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy.stats import qmc
import torch, torch.nn as nn
import DSGRN, DSGRN_utils
np.set_printoptions(precision=3, suppress=True)

# Train on a GPU when one is available: CUDA on Colab, MPS on Apple silicon, otherwise CPU.
# Note that MPS is single precision only, so CUDA is set to float32 for reproducibility.
if torch.cuda.is_available():
    DEVICE, TORCH_DTYPE = torch.device('cuda'), torch.float32
elif torch.backends.mps.is_available():
    DEVICE, TORCH_DTYPE = torch.device('mps'), torch.float32
else:
    DEVICE, TORCH_DTYPE = torch.device('cpu'), torch.float64
torch.set_default_dtype(TORCH_DTYPE)
print('torch device:', DEVICE, '| dtype:', TORCH_DTYPE)

In [ ]:
# ---- config: single source of truth for every hyperparameter ----
from dataclasses import dataclass

@dataclass
class HParams:
    # model
    HIDDEN_WIDTH: int = 64
    DEPTH: int = 4
    FOURIER_K: int = 5          # Fourier feature-map width (Ex3's periodic solution needs it)
    USE_FEATURE_MAP: bool = True  # toggle the feature map on/off without touching FOURIER_K
    # optimizer (AdamW; WEIGHT_DECAY=0 => plain Adam. Decay never touches `raw`.)
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 0.0
    MAX_EPOCHS: int = 8000
    LBFGS_ITERS: int = 0        # 0 = skip L-BFGS phase
    # loss weights
    W_DATA: float = 50.0
    W_PHYS: float = 10.0
    W_IC: float = 1.0
    # sweep
    NOISE_LEVELS: tuple = (0.0, 0.01, 0.05, 0.10, 0.25, 0.50)
    N_TRIALS: int = 15
    SEED_NOISE_BASE: int = 100
    SEED_INIT_BASE: int = 0

HP = HParams()
M = 3

### Hill functions

The production is built from Hill functions
\begin{equation}
f^+(x,L,U,\theta,d) = L + (U-L) \frac{x^d}{x^d+\theta^d}, \quad
f^-(x,L,U,\theta,d) = L + (U-L) \frac{\theta^d}{x^d+\theta^d}
\end{equation}
where
*   `L` and `U` are lower and upper values,
*   `theta` is a threshold value,
*   `d` is the steepness coefficient.

We use `f+` when it is activating and `f-` when it is repressing, noting that $f^++f^-=L+U$.

We define `hill_act` and `hill_rep` below, both in NumPy for simulation and in torch for the PINN's physics loss.

In [ ]:
# --- Hill production functions: smooth switches (gamma normalized to 1) ---
def hill_act(x, L, U, th, d):   # activating: low L -> high U as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * xd / (td + xd)

def hill_rep(x, L, U, th, d):   # repressing: high U -> low L as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * td / (td + xd)

In [ ]:
# --- the same two functions in torch (used inside the network's physics loss) ---
def hill_act_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * xd / (td + xd)

def hill_rep_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * td / (td + xd)

## 1. Model and DSGRN region (node 13)

Consider the three-gene repression cycle ($\gamma=1$), `x3 -| x1 -| x2 -| x3`,

$$\dot{x}_1=-x_1+f^-(x_3),\quad \dot{x}_2=-x_2+f^-(x_1),\quad \dot{x}_3=-x_3+f^-(x_2).$$

Edges are indexed `[source, target]`. For node $j$ the DSGRN region is set by ordering its production levels $L_{ij}, U_{ij}$ (from the input $i\to j$) against its **output** threshold $\gamma_j\theta_{jk}$ (the edge $j\to k$ it drives), not against its own input threshold. Node 13 is the oscillatory region, where every node straddles its output threshold:
$$L_{31}<\gamma_1\theta_{12}<U_{31},\qquad L_{12}<\gamma_2\theta_{23}<U_{12},\qquad L_{23}<\gamma_3\theta_{31}<U_{23}.$$
Membership is tested with DSGRN: `to_matrices` packs `(L, U, theta)` into DSGRN's `[source, target]` matrices (with `T`$=\gamma\,\theta$), `par_index_from_sample` returns the region index, and `in_region` checks it against node 13.

In [ ]:
# Ground-truth parameters sampled from DSGRN node 13 (Wael's set, param_ex3.csv row 0).
# Edges are labelled [source][target]:  z->x = "31" [2,0],  x->y = "12" [0,1],  y->z = "23" [1,2].
P = dict(L31=0.873, U31=1.479, th31=0.682, d31=15.0,   # z -> x
         L12=0.022, U12=1.253, th12=1.139, d12=15.0,   # x -> y
         L23=0.121, U23=1.840, th23=0.850, d23=15.0,   # y -> z
         g=1.0)

# Earlier symmetric representative point, kept for future work. Every edge shares the same
# (L, U, theta), so it cannot distinguish the [source, target] packing from its transpose --
# a weak test of the DSGRN labelling. The asymmetric sample above is a real test.
# P = dict(L31=1.0,U31=5.0,th31=3.0,d31=10.0, L12=1.0,U12=5.0,th12=3.0,d12=10.0,
#          L23=1.0,U23=5.0,th23=3.0,d23=10.0, g=1.0)

def rhs(t, x, p):
    x1, x2, x3 = x
    return [-p['g']*x1 + hill_rep(x3, p['L31'],p['U31'],p['th31'],p['d31']),
            -p['g']*x2 + hill_rep(x1, p['L12'],p['U12'],p['th12'],p['d12']),
            -p['g']*x3 + hill_rep(x2, p['L23'],p['U23'],p['th23'],p['d23'])]

NET_SPEC = 'x : ~z\ny : ~x\nz : ~y\n'
_net = DSGRN.Network(NET_SPEC); _pg = DSGRN.ParameterGraph(_net)
TARGET = 13
def to_matrices(p):              # (L, U, theta) -> DSGRN L, U, T matrices, all indexed [source, target]
    # DSGRN entry T[i,j] = gamma_i * theta_{ij} (threshold on the source x_i; gamma = 1, so T = theta).
    # Node j is tested by ordering its production {L[i,j], U[i,j]} (inputs i->j) against its
    # OUTPUT thresholds T[j,k] (edges j->k):  L_{ij} < gamma_j theta_{jk} < U_{ij}.
    n = 3
    L = np.zeros((n, n)); U = np.zeros((n, n)); T = np.zeros((n, n))
    L[2,0],U[2,0]=p['L31'],p['U31']; L[0,1],U[0,1]=p['L12'],p['U12']; L[1,2],U[1,2]=p['L23'],p['U23']
    T[2,0],T[0,1],T[1,2]=p['th31'],p['th12'],p['th23']
    return L, U, T
def region_of(p):                # parameters -> DSGRN region index (-1 if outside)
    return DSGRN.par_index_from_sample(_pg, *to_matrices(p))
def in_region(p):                # membership in the target region
    return region_of(p) == TARGET

print('do the true parameters lie in node 13?', in_region(P), '| region', region_of(P))

In [ ]:
def simulate(rhs, p, x0, T, n):
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(p,), rtol=1e-9, atol=1e-11)
    return t, sol.y.T   # times (n,), states (n, m)

def add_noise(y, ub_frac, scale, rng):
    yn = y + rng.uniform(-ub_frac*scale, ub_frac*scale, size=y.shape)
    return np.clip(yn, 0.0, None)   # concentrations stay non-negative

## 2. Synthetic data

Ground truth parameters were chosen in node 13. The system is integrated over a few time units (about one oscillation period), then corrupted with bounded noise.

In [ ]:
T, n = 15.0, 200   # window matches Wael's Tfinal for these parameters
MARGIN = 1.2
box_hi = [MARGIN*P['U31'], MARGIN*P['U12'], MARGIN*P['U23']]
n_ic = 6
x0s = qmc.scale(qmc.LatinHypercube(d=3, seed=0, optimization='random-cd').random(n_ic), [0.0, 0.0, 0.0], box_hi).tolist()
ts, xs = [], []
for x0 in x0s:
    t, y = simulate(rhs, P, x0, T, n); ts.append(t); xs.append(y)

gap = min(abs(P['L31']-P['U31']), abs(P['L12']-P['U12']), abs(P['L23']-P['U23']))
rng = np.random.default_rng(0)
xs_noisy = [add_noise(y, 0.25, gap, rng) for y in xs]

plt.figure(figsize=(8, 3.2))
plt.plot(ts[0], xs[0][:,0], label='$x_1$'); plt.plot(ts[0], xs[0][:,1], label='$x_2$')
plt.plot(ts[0], xs[0][:,2], label='$x_3$')
plt.plot(ts[0], xs_noisy[0][:,0], '.', ms=3, color='C0', alpha=.5)
plt.xlabel('t'); plt.legend(); plt.title('Repressilator (one trajectory, with noise)')
plt.show()

## 3. Least squares and region check

Least squares fits the nine `(L, U, theta)` and three per-edge `d` from finite-difference slopes, with `gamma` fixed. Only the nine `(L, U, theta)` are scored. Region membership (node 13) is checked across a noise sweep.

In [ ]:
KEYS = ['L31','U31','th31','L12','U12','th12','L23','U23','th23']
DKEYS = ['d31','d12','d23']
def ls_recover(ts, xs, g=1.0):
    X = np.vstack(xs)
    DX = np.vstack([np.gradient(y, t, axis=0) for t, y in zip(ts, xs)])
    scale = np.maximum(np.std(DX, axis=0), 1e-6)
    def resid(z):
        p = dict(zip(KEYS, z[:9])); p.update(dict(zip(DKEYS, z[9:])))
        f1 = -g*X[:,0] + hill_rep(X[:,2],p['L31'],p['U31'],p['th31'],p['d31'])
        f2 = -g*X[:,1] + hill_rep(X[:,0],p['L12'],p['U12'],p['th12'],p['d12'])
        f3 = -g*X[:,2] + hill_rep(X[:,1],p['L23'],p['U23'],p['th23'],p['d23'])
        return np.concatenate([(DX[:,0]-f1)/scale[0], (DX[:,1]-f2)/scale[1], (DX[:,2]-f3)/scale[2]])
    z0 = np.append(np.array([1.,5.,3., 1.,5.,3., 1.,5.,3.]) * 0.8, [8.0, 8.0, 8.0])
    s = least_squares(resid, z0, bounds=([0]*9+[0.25]*3, [30]*9+[80]*3),
                      max_nfev=20000, loss='soft_l1')
    p = dict(zip(KEYS, s.x[:9])); p.update(dict(zip(DKEYS, s.x[9:]))); p['g'] = g; return p

p_ls = ls_recover(ts, xs_noisy)
print('LS estimate:', {k: round(p_ls[k],2) for k in KEYS})
print('LS lands in node 13:', in_region(p_ls))

levels = [0.0, 0.1, 0.25, 0.5]; rate = []
for ub in levels:
    r = np.random.default_rng(11); hits = 0
    for _ in range(15):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        hits += in_region(ls_recover(ts, xs_n))
    rate.append(hits/15)
print('LS region-recovery rate by noise:', dict(zip(levels, rate)))

In [ ]:
def build_tensors(ts, xs, x0s):
    # flatten all trajectories into (N,1) time, (N,m) repeated x0, (N,m) measured states
    t_d = np.concatenate([t[:, None] for t in ts])
    x_d = np.vstack(xs)
    x0_d = np.vstack([np.tile(x0, (len(t), 1)) for x0, t in zip(x0s, ts)])
    t_ic = np.zeros((len(x0s), 1)); x_ic = np.array(x0s, dtype=float)  # t=0 anchors
    to = lambda a: torch.tensor(a, dtype=torch.get_default_dtype(), device=DEVICE)
    return (to(t_d), to(x0_d), to(x_d), to(t_ic), to(x_ic))

## 4. PINN with Fourier feature mapping

The surrogate receives Fourier features of time (`sin`/`cos` at several frequencies, `fourier_k = 5`) to represent the periodic solution; `fourier_k = 0` reduces to a plain time input. The PINN jointly learns `(L, U, theta, d)` from a neutral init, with `gamma` fixed at 1. Only `(L, U, theta)` are scored.

In [ ]:
# --- the physics-informed network: surrogate u_theta(t, x0) + learnable (L,U,theta,d) ---
class PINN(nn.Module):
    def __init__(self, m, param_init, T, hidden=64, depth=4, fourier_k=0):
        super().__init__()
        self.m, self.T, self.fourier_k = m, T, fourier_k
        in_dim = (2*fourier_k if fourier_k > 0 else 1) + m
        if fourier_k > 0:
            self.register_buffer('freqs', 2*np.pi*torch.arange(1, fourier_k+1, dtype=torch.get_default_dtype()))
        layers, d0 = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d0, hidden), nn.Tanh()]; d0 = hidden
        layers += [nn.Linear(d0, m)]
        self.net = nn.Sequential(*layers)
        # physical parameters via positive reparametrization  p = raw^2 + eps
        self.raw = nn.ParameterDict(
            {k: nn.Parameter(torch.tensor(float(v)**0.5)) for k, v in param_init.items()})

    def phys_params(self):
        return {k: self.raw[k]**2 + 1e-6 for k in self.raw}

    def _feat(self, t):
        tn = t / self.T   # normalize time to [0,1]; the chain rule is handled by autograd
        if self.fourier_k > 0:
            return torch.cat([torch.sin(self.freqs*tn), torch.cos(self.freqs*tn)], dim=1)
        return tn

    def forward(self, t, x0):
        return self.net(torch.cat([self._feat(t), x0], dim=1))

def fit_pinn(model, data, rhs_t, hp, log=0):
    t_d, x0_d, x_d, t_ic, x_ic = data
    def total_loss():
        loss_data = ((model(t_d, x0_d) - x_d)**2).mean()
        tc = t_d.clone().requires_grad_(True)
        u = model(tc, x0_d)
        grads = [torch.autograd.grad(u[:, j].sum(), tc, create_graph=True)[0]
                 for j in range(model.m)]
        du = torch.cat(grads, dim=1)
        loss_phys = ((du - rhs_t(u, model.phys_params()))**2).mean()
        loss_ic = ((model(t_ic, x_ic) - x_ic)**2).mean()
        return hp.W_DATA*loss_data + hp.W_PHYS*loss_phys + hp.W_IC*loss_ic
    # decoupled decay: only network weights, never the raw physical params
    opt = torch.optim.AdamW(
        [{'params': model.net.parameters(), 'weight_decay': hp.WEIGHT_DECAY},
         {'params': model.raw.parameters(), 'weight_decay': 0.0}],
        lr=hp.LEARNING_RATE)
    for it in range(hp.MAX_EPOCHS):
        opt.zero_grad(); loss = total_loss(); loss.backward(); opt.step()
        if log and it % log == 0:
            print(f'  step {it:5d}  loss {loss.item():.2e}')
    if hp.LBFGS_ITERS > 0:
        lb = torch.optim.LBFGS(model.parameters(), max_iter=hp.LBFGS_ITERS,
                               line_search_fn='strong_wolfe')
        lb.step(lambda: (lb.zero_grad(), total_loss().backward(), total_loss())[-1])
    return model

In [ ]:
def rhs_t(x, p):
    x1, x2, x3 = x[:,0:1], x[:,1:2], x[:,2:3]
    d1 = -x1 + hill_rep_t(x3, p['L31'],p['U31'],p['th31'],p['d31'])
    d2 = -x2 + hill_rep_t(x1, p['L12'],p['U12'],p['th12'],p['d12'])
    d3 = -x3 + hill_rep_t(x2, p['L23'],p['U23'],p['th23'],p['d23'])
    return torch.cat([d1, d2, d3], dim=1)

init = {k: (0.05 if k[0]=='L' else 1.0) for k in KEYS}
init.update({k: 5.0 for k in DKEYS})
torch.manual_seed(0)
fk = HP.FOURIER_K if HP.USE_FEATURE_MAP else 0   # feature map on by default; HP.USE_FEATURE_MAP=False ablates it
model = PINN(m=M, param_init=init, T=T, hidden=HP.HIDDEN_WIDTH, depth=HP.DEPTH, fourier_k=fk).to(DEVICE)
data = build_tensors(ts, xs_noisy, x0s)
model = fit_pinn(model, data, rhs_t, HP, log=500)

pp = {k: float(v.detach()) for k, v in model.phys_params().items()}
print('PINN estimate:', {k: round(pp[k],2) for k in KEYS})
print('PINN lands in node 13:', in_region(pp))

In [ ]:
# ---- unified sweep + reporting (notebook-independent) ----
def run_sweep(recover_fn, clean_xs, scale, membership_fn, hp):
    """recover_fn(xs_noisy, init_seed) -> params dict. Returns {noise: hit_count}."""
    rates = {}
    for ub in hp.NOISE_LEVELS:
        hits = 0
        for k in range(hp.N_TRIALS):
            rng = np.random.default_rng(hp.SEED_NOISE_BASE + k)
            xs_n = [add_noise(y, ub, scale, rng) for y in clean_xs]
            hits += int(membership_fn(recover_fn(xs_n, hp.SEED_INIT_BASE + k)))
        rates[ub] = hits
    return rates

def sweep_table(pinn_rates, ls_rates, hp, title):
    N = hp.N_TRIALS
    print(f'{"noise":>6} | {"PINN":>8} | {"LS":>8}')
    px, pp_, lp_ = [], [], []
    for ub in hp.NOISE_LEVELS:
        pr, lr = pinn_rates[ub], ls_rates[ub]
        print(f'{ub*100:>5.0f}% | {pr:>3}/{N} | {lr:>3}/{N}')
        px.append(ub*100); pp_.append(pr/N); lp_.append(lr/N)
    plt.plot(px, pp_, 'o-', label='PINN')
    plt.plot(px, lp_, 's--', label='least squares')
    plt.xlabel('noise upper bound (%)'); plt.ylabel('region-recovery rate')
    plt.ylim(-0.05, 1.05); plt.legend(); plt.title(title); plt.show()

def pinn_recover(xs_n, init_seed):
    torch.manual_seed(init_seed)
    fk = HP.FOURIER_K if HP.USE_FEATURE_MAP else 0
    model = PINN(m=M, param_init=init, T=T, hidden=HP.HIDDEN_WIDTH, depth=HP.DEPTH, fourier_k=fk).to(DEVICE)
    model = fit_pinn(model, build_tensors(ts, xs_n, x0s), rhs_t, HP)
    return {k: float(v.detach()) for k, v in model.phys_params().items()}

## 5. Region recovery across noise (PINN vs least squares)

For each noise level (% of total bound), both methods are repeated over independent noisy datasets and the fraction landing in node 13 is reported. This is the feature-map ablation from the paper: with `HP.USE_FEATURE_MAP=True` (default) the PINN recovers node 13 in most trials; setting `HP.USE_FEATURE_MAP=False` reproduces the collapse reported without the Fourier feature map.

In [ ]:
# set HP.USE_FEATURE_MAP = False to reproduce the no-feature-map ablation (region recovery collapses)
ls_rates   = run_sweep(lambda xs_n, s: ls_recover(ts, xs_n), xs, gap, in_region, HP)
pinn_rates = run_sweep(pinn_recover, xs, gap, in_region, HP)
sweep_table(pinn_rates, ls_rates, HP, 'Repressilator: region recovery vs noise')

## Morse graph and Conley-index recovery (DSGRN)

There are two criteria per recovered parameter set: exact DSGRN region equality, and label-preserving isomorphism of the **Conley-Morse graph** (recurrent Morse sets, reachability order, Conley-index labels). Region equality implies isomorphic Morse graphs, so the second criterion is weaker than the first.

The cells below compare the recovered and target Morse graphs via `par_index_from_sample` (parameter sample to region index), `DSGRN_utils.ConleyMorseGraph` (region index to Morse graph), and `DSGRN.isomorphic_morse_graphs` (isomorphism of Morse graphs). For the target region we also plot the Morse sets in state space with `DSGRN_utils.PlotMorseSets` and the Conley-Morse graph with `DSGRN_utils.PlotMorseGraph`.

In [ ]:
# Conley-Morse graph for each region, reusing the network and TARGET from Section 1
_mg = {}
def conley_morse(idx):          # region index -> Conley-Morse graph (cached)
    if idx not in _mg:
        _mg[idx] = DSGRN_utils.ConleyMorseGraph(_pg.parameter(idx))[0]
    return _mg[idx]
def morse_recovers(idx, target=TARGET):   # same Conley-Morse graph up to label-preserving iso
    return idx >= 0 and DSGRN.isomorphic_morse_graphs(conley_morse(idx), conley_morse(target))

_cmg = DSGRN_utils.ConleyMorseGraph(_pg.parameter(TARGET))

In [ ]:
DSGRN_utils.PlotMorseSets(*_cmg, proj_dims=[0, 1])

In [ ]:
DSGRN_utils.PlotMorseGraph(_cmg[0])

In [ ]:
# the recovered parameters from the cells above
p_ls = ls_recover(ts, xs_noisy)
print('recovered region index:', region_of(p_ls),
      '| exact region match:', region_of(p_ls) == TARGET,
      '| Morse/Conley match:', morse_recovers(region_of(p_ls)))

# noise sweep: exact-region recovery vs Morse/Conley recovery (least squares)
levels = [0.0, 0.1, 0.25, 0.5]
print(f'{"noise":>6} | {"exact region":>12} | {"Morse/Conley":>12} | {"Morse-ok / miss":>16}')
for ub in levels:
    r = np.random.default_rng(7); reg = mor = gapc = 0
    for _ in range(15):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        idx = region_of(ls_recover(ts, xs_n))
        er = (idx == TARGET); mr = morse_recovers(idx)
        reg += er; mor += mr; gapc += (mr and not er)
    miss = 15 - reg
    print(f'{ub:>6} | {reg:>10}/15 | {mor:>10}/15 | {f"{gapc}/{miss}":>16}')